In [27]:
import multiprocessing
import concurrent.futures
import time
import random
from multiprocessing import Semaphore

In [28]:
# 1. Square Program
def square(n):
    return n * n

def sequential_square(numbers):
    return [square(n) for n in numbers]

def multiprocessing_square(numbers):
    num_workers = 8
    chunk_size = len(numbers) // (num_workers * 2)  
    with multiprocessing.Pool(processes=num_workers) as pool:
        return pool.map(square, numbers, chunksize=chunk_size)

def multiprocessing_pool_map(numbers):
    with multiprocessing.Pool() as pool:
        return pool.map(square, numbers)

def multiprocessing_pool_apply(numbers):
    with multiprocessing.Pool() as pool:
        return [pool.apply(square, (n,)) for n in numbers]

def concurrent_futures_square(numbers):
    with concurrent.futures.ProcessPoolExecutor() as executor:
        return list(executor.map(square, numbers))

def run_square_tests(size):
    numbers = [random.randint(1, 100) for _ in range(size)]
    
    for method in [sequential_square, multiprocessing_square, multiprocessing_pool_map, multiprocessing_pool_apply, concurrent_futures_square]:
        start_time = time.time()
        method(numbers)
        print(f"{method.__name__}: {time.time() - start_time:.4f} sec")


In [29]:
# 2. Process Synchronization with Semaphores
class ConnectionPool:
    def __init__(self, size):
        self.size = size
        self.semaphore = Semaphore(size)
        self.connections = [f"Connection {i}" for i in range(size)]

    def get_connection(self):
        self.semaphore.acquire()
        return self.connections.pop()

    def release_connection(self, conn):
        self.connections.append(conn)
        self.semaphore.release()

In [30]:
def access_database(pool, process_id):
    print(f"Process {process_id} waiting for connection...")
    conn = pool.get_connection()
    print(f"Process {process_id} acquired {conn}")
    time.sleep(random.uniform(0.5, 2))  # Simulate database operation
    print(f"Process {process_id} releasing {conn}")
    pool.release_connection(conn)

def run_connection_pool_test(pool_size, process_count):
    pool = ConnectionPool(pool_size)
    processes = [multiprocessing.Process(target=access_database, args=(pool, i)) for i in range(process_count)]
    for p in processes:
        p.start()
    for p in processes:
        p.join()

In [24]:
if __name__ == "__main__":
    print("Testing Square Program with 10^6 numbers:")
    run_square_tests(10**6)
    print("\nTesting Square Program with 10^7 numbers:")
    run_square_tests(10**7)
    
    print("\nTesting Connection Pool Synchronization:")
    run_connection_pool_test(pool_size=3, process_count=6)


Testing Square Program with 10^6 numbers:
sequential_square: 0.0633 sec
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
  File "/tmp/ipykernel_179566/173459883.py", line 3, in <module>
    run_square_tests(10**6)
  File "/tmp/ipykernel_179566/357465185.py", line 33, in run_square_tests
    method(numbers)
  File "/tmp/ipykernel_179566/357465185.py", line 11, in multiprocessing_square
    with multiprocessing.Pool(processes=num_workers) as pool:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/context.py", line 119, in Pool
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/pool.py", line 191, in __init__
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/pool.py", line 346, in _setup_queues
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/context.py", line 113, in SimpleQueue
  Fi

In [33]:
import random
import time
import multiprocessing
import concurrent.futures


# --------------------------- Task 1: Square Program ---------------------------

# Function to compute the square of a number
def square(n):
    return n ** 2


# Create a list of 10^6 random numbers
numbers = [random.randint(1, 100) for _ in range(10**6)]


# Function to process smaller chunks
def process_chunk(chunk):
    return [square(n) for n in chunk]


# Timing the program in different scenarios

# Sequential for loop
start_time = time.time()
squares_sequential = [square(n) for n in numbers]
print(f"Sequential time: {time.time() - start_time} seconds")


# Multiprocessing pool with map() for chunks (process in smaller chunks)
def chunked_multiprocessing():
    chunk_size = 10000  # Process in chunks of 10,000
    chunks = [numbers[i:i + chunk_size] for i in range(0, len(numbers), chunk_size)]
    
    start_time = time.time()
    with multiprocessing.Pool(processes=4) as pool:
        results = pool.map(process_chunk, chunks)
    print(f"Chunked Multiprocessing Pool with map() time: {time.time() - start_time} seconds")


chunked_multiprocessing()


# Multiprocessing pool with apply() for chunks (process in smaller chunks)
def chunked_multiprocessing_apply():
    chunk_size = 10000  # Process in chunks of 10,000
    chunks = [numbers[i:i + chunk_size] for i in range(0, len(numbers), chunk_size)]

    start_time = time.time()
    with multiprocessing.Pool(processes=4) as pool:
        results = [pool.apply(process_chunk, args=(chunk,)) for chunk in chunks]
    print(f"Chunked Multiprocessing Pool with apply() time: {time.time() - start_time} seconds")


chunked_multiprocessing_apply()


# ----------------------- Task 2: Process Synchronization with Semaphores -----------------------

class ConnectionPool:
    def __init__(self, size):
        self.semaphore = multiprocessing.Semaphore(size)  # Limit the number of connections
        self.connections = [f"Connection {i}" for i in range(size)]

    def get_connection(self):
        self.semaphore.acquire()
        connection = self.connections.pop()
        print(f"Connection {connection} acquired.")
        return connection

    def release_connection(self, connection):
        self.connections.append(connection)
        print(f"Connection {connection} released.")
        self.semaphore.release()


# Simulate a database operation
def access_database(pool):
    connection = pool.get_connection()
    time.sleep(random.uniform(0.1, 1))  # Simulating work with random sleep
    pool.release_connection(connection)


# Set up multiprocessing to simulate access to the connection pool
def main():
    pool = ConnectionPool(size=3)  # Limited to 3 connections
    processes = []
    
    for _ in range(10):  # Simulate 10 processes trying to access the pool
        p = multiprocessing.Process(target=access_database, args=(pool,))
        processes.append(p)
        p.start()
    
    for p in processes:
        p.join()


if __name__ == "__main__":
    # Run the multiprocessing for the square program
    print("\n--- Square Program Timings ---")
    # Add the sequential and multiprocessing code above here

    # Run the Connection Pool test
    print("\n--- Process Synchronization with Semaphores ---")
    main()


Sequential time: 0.07471442222595215 seconds
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
  File "/tmp/ipykernel_179566/3455594377.py", line 42, in <module>
    chunked_multiprocessing()
  File "/tmp/ipykernel_179566/3455594377.py", line 37, in chunked_multiprocessing
    with multiprocessing.Pool(processes=4) as pool:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/context.py", line 119, in Pool
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/pool.py", line 191, in __init__
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/pool.py", line 346, in _setup_queues
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/context.py", line 113, in SimpleQueue
  File "/home/student/anaconda3/envs/parallel/lib/python3.12/multiprocessing/queues.py", line 362, in __init__
 